In [3]:
import pandas as pd
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModelForCausalLM, AutoTokenizer, EsmModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Compute Device: {device.type.upper()}")

if device.type == 'cuda':
    print(f"  • GPU: {torch.cuda.get_device_name(0)}")

c:\Users\win_10\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Active Compute Device: CUDA
  • GPU: NVIDIA GeForce GTX 960M


**Load sequence embeddings and multimodal profiles**

In [4]:
# THE RETRIEVAL SPACE (The text and clusters to feed the LLM)
multimodal_df = pd.read_csv('../data/processed/final_multimodal_clusters.csv')
known_ids = multimodal_df['ID'].tolist()

# THE SEARCH SPACE (The pure ESM-2 sequence embeddings)
seq_embeddings_df = pd.read_csv('../data/processed/embedding_sequences_mean_pooling.csv')
if 'uniref_id' in seq_embeddings_df.columns:
    seq_embeddings_df = seq_embeddings_df.rename(columns={'uniref_id': 'ID'})

# Filter and align the sequence embeddings exactly to the known IDs
pure_seq_df = seq_embeddings_df[seq_embeddings_df['ID'].isin(known_ids)].copy()
pure_seq_df = pure_seq_df.set_index('ID').reindex(known_ids)

# Extract the embedding vectors
embedding_cols = [col for col in pure_seq_df.columns if col != 'ID']
known_sequence_vectors = pure_seq_df[embedding_cols].values

print(f"Loaded {len(known_ids)} Multimodal Profiles.")
print(f"Loaded {known_sequence_vectors.shape} Sequence Search Matrix.")
print(f"Embedding dimension: {known_sequence_vectors.shape[1]}")

if known_sequence_vectors.shape[0] != len(known_ids):
    print(f"WARNING: Mismatch between multimodal ({len(known_ids)}) and embeddings ({known_sequence_vectors.shape[0]})")
else:
    print(f"Data alignment verified!")

Loaded 1037 Multimodal Profiles.
Loaded (1037, 1280) Sequence Search Matrix.
Embedding dimension: 1280
Data alignment verified!


**Load ESM-2 Protein Language Model**

In [6]:
# Clear GPU cache before loading model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB total")
    print(f"Available: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9:.2f} GB\n")

esm_model_id = "facebook/esm2_t33_650M_UR50D"
esm_tokenizer = AutoTokenizer.from_pretrained(esm_model_id)
esm_model = EsmModel.from_pretrained(
    esm_model_id,
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    device_map="auto" if device.type == 'cuda' else None
)
if device.type != 'cuda':
    esm_model = esm_model.to(device)
esm_model.eval() 
print("ESM-2 Biological Encoder loaded on GPU.")

if torch.cuda.is_available():
    print(f"   GPU Memory Used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"   GPU Memory Free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9:.2f} GB")

GPU Memory: 4.29 GB total
Available: 2.99 GB



Loading weights: 100%|██████████| 566/566 [00:02<00:00, 217.36it/s, Materializing param=encoder.layer.32.output.dense.weight]                      
EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ESM-2 Biological Encoder loaded on GPU.
   GPU Memory Used: 1.30 GB
   GPU Memory Free: 2.99 GB


**Build Cluster Knowledge Base**

In [12]:
print("Building Cluster Knowledge Base from 1037 labeled proteins...")
print("="*70)

cluster_kb = {}

for cluster_id in sorted(multimodal_df['super_cluster'].unique()):
    cluster_proteins = multimodal_df[multimodal_df['super_cluster'] == cluster_id]

    names, keywords, functions, subfamilies, go_terms = [], [], [], [], []

    for _, row in cluster_proteins.iterrows():
        if pd.notna(row.get('Protein names', '')):
            n = str(row['Protein names']).strip()
            if n:
                names.append(n[:120])
        if pd.notna(row.get('Keywords', '')):
            keywords.extend([k.strip() for k in str(row['Keywords']).split(';') if k.strip()])
        if pd.notna(row.get('Function [CC]', '')):
            f = str(row['Function [CC]']).strip()
            if f:
                functions.append(f[:300])
        if pd.notna(row.get('subfamily', '')):
            s = str(row['subfamily']).strip()
            if s:
                subfamilies.append(s)
        if pd.notna(row.get('Gene Ontology (GO)', '')):
            go_terms.extend([g.strip() for g in str(row['Gene Ontology (GO)']).split(';') if g.strip()])

    cluster_kb[int(cluster_id)] = {
        'size': len(cluster_proteins),
        'subfamilies': list(dict.fromkeys(subfamilies))[:6],    
        'keywords': list(dict.fromkeys(keywords))[:20],
        'sample_functions': list(dict.fromkeys(functions))[:3],
        'sample_names': list(dict.fromkeys(names))[:5],
        'go_terms': list(dict.fromkeys(go_terms))[:10],
    }

print(f"Knowledge base built for {len(cluster_kb)} clusters")
print(f"\nCluster sizes (top 10):")
for cid, info in sorted(cluster_kb.items(), key=lambda x: -x[1]['size'])[:10]:
    label = "NOISE" if cid == -1 else f"Cluster {cid}"
    print(f"   {label:15s}: {info['size']} proteins | subfamilies: {info['subfamilies'][:3]}")

print(f"\nSample — Cluster 0:")
c0 = cluster_kb[0]
print(f"   Size     : {c0['size']}")
print(f"   Subfamilies: {c0['subfamilies']}")
print(f"   Keywords   : {c0['keywords'][:8]}")
print(f"   Function   : {c0['sample_functions'][0][:200] if c0['sample_functions'] else 'N/A'}")


Building Cluster Knowledge Base from 1037 labeled proteins...
Knowledge base built for 59 clusters

Cluster sizes (top 10):
   NOISE          : 272 proteins | subfamilies: ['Other_ClassA', 'Adrenergic_beta', 'Adrenergic_alpha']
   Cluster 38     : 37 proteins | subfamilies: ['Adrenergic_beta']
   Cluster 29     : 34 proteins | subfamilies: ['Adrenergic_beta']
   Cluster 53     : 27 proteins | subfamilies: ['Other_ClassA']
   Cluster 39     : 25 proteins | subfamilies: ['Adrenergic_beta']
   Cluster 49     : 25 proteins | subfamilies: ['Other_ClassA']
   Cluster 11     : 24 proteins | subfamilies: ['Adrenergic_beta']
   Cluster 50     : 24 proteins | subfamilies: ['Other_ClassA']
   Cluster 12     : 23 proteins | subfamilies: ['Other_ClassA']
   Cluster 42     : 20 proteins | subfamilies: ['Other_ClassA']

Sample — Cluster 0:
   Size     : 17
   Subfamilies: ['Adrenergic_alpha']
   Keywords   : ['Cell membrane', 'Cytoplasm', 'Disulfide bond', 'G-protein coupled receptor', 'Glycoprotein'

**Load Language Model (TinyLlama Chat)**

In [17]:
# Free ESM-2 from GPU memory so the LLM has room to load
if device.type == 'cuda' and 'esm_model' in dir():
    try:
        del esm_model
        torch.cuda.empty_cache()
        print("ESM-2 unloaded from GPU to free memory.")
    except Exception:
        pass

llm_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading LLM: {llm_model_id}")
llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_id)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_id,
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
    device_map="auto" if device.type == 'cuda' else None,
)
if device.type != 'cuda':
    llm_model = llm_model.to(device)
llm_model.eval()

print(f"LLM loaded — {llm_model_id}")
if device.type == 'cuda':
    print(f"   GPU memory used: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
    print(f"   GPU memory free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.2f} GB")


ESM-2 unloaded from GPU to free memory.
Loading LLM: TinyLlama/TinyLlama-1.1B-Chat-v1.0


Loading weights: 100%|██████████| 201/201 [00:05<00:00, 39.55it/s, Materializing param=model.norm.weight]                              


LLM loaded — TinyLlama/TinyLlama-1.1B-Chat-v1.0
   GPU memory used: 1.20 GB
   GPU memory free: 3.10 GB


**RAG Pipeline: Sequence → Retrieve → LLM Answer**

In [19]:
from collections import Counter
import re as _re

def clean_sequence(raw: str) -> str:
    seq = raw.replace('-', '').replace('.', '')
    seq = seq.upper()
    return seq


def _get_esm_model():
    g = globals()
    if 'esm_model' in g and g['esm_model'] is not None:
        return g['esm_model'], g['esm_tokenizer']
    print("Reloading ESM-2 for embedding ...")
    from transformers import EsmModel, AutoTokenizer as AT
    _tok = AT.from_pretrained("facebook/esm2_t33_650M_UR50D")
    _mdl = EsmModel.from_pretrained(
        "facebook/esm2_t33_650M_UR50D",
        torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
        device_map="auto" if device.type == 'cuda' else None,
    )
    if device.type != 'cuda':
        _mdl = _mdl.to(device)
    _mdl.eval()
    g['esm_model'] = _mdl
    g['esm_tokenizer'] = _tok
    return _mdl, _tok


def embed_sequence(raw_sequence: str) -> np.ndarray:

    mdl, tok = _get_esm_model()

    seq = clean_sequence(raw_sequence)
    if len(seq) == 0:
        raise ValueError("Sequence is empty after cleaning — check input.")

    seq = seq[:1022]
    inputs = tok(seq, return_tensors="pt", add_special_tokens=True).to(device)
    with torch.no_grad():
        outputs = mdl(**inputs)
    token_embs = outputs.last_hidden_state[0, 1:-1, :]
    vec = token_embs.mean(dim=0).float().cpu().numpy()
    return vec.reshape(1, -1)


def retrieve_neighbors(query_vec: np.ndarray, k: int = 3):
    sims = cosine_similarity(query_vec, known_sequence_vectors)[0]
    top_k = np.argsort(sims)[::-1][:k]
    return [
        {
            'rank': i + 1,
            'idx': int(top_k[i]),
            'id': known_ids[top_k[i]],
            'similarity': float(sims[top_k[i]]),
            'data': multimodal_df.iloc[top_k[i]],
        }
        for i in range(k)
    ]


def build_rag_context(neighbors: list):
    clusters = [int(n['data']['super_cluster']) for n in neighbors]
    vote_counts = Counter(clusters)
    voted_cluster = vote_counts.most_common(1)[0][0]
    confidence = vote_counts[voted_cluster] / len(clusters)

    fallback_used = False
    if voted_cluster == -1:
        non_noise = [c for c in clusters if c != -1]
        if non_noise:
            voted_cluster = Counter(non_noise).most_common(1)[0][0]
            confidence = non_noise.count(voted_cluster) / len(clusters)
            fallback_used = True

    kb = cluster_kb.get(voted_cluster, {})
    label = "NOISE / unclassified" if voted_cluster == -1 else f"Cluster {voted_cluster}"

    lines = [
        f"=== PREDICTED GPCR CLUSTER: {label} ===",
    ]
    if fallback_used:
        lines.append(
            f"(Note: majority vote was cluster -1/noise; using best non-noise cluster "
            f"from neighbors — {confidence:.0%} support)\n"
        )
    else:
        lines.append(
            f"(Majority vote: {vote_counts[voted_cluster]}/{len(clusters)} neighbors "
            f"agree — {confidence:.0%} confidence)\n"
        )

    lines += [
        "--- CLUSTER BIOLOGICAL PROFILE ---",
        f"Subfamilies  : {', '.join(kb.get('subfamilies', [])) or 'N/A'}",
        f"Keywords     : {', '.join(kb.get('keywords', [])[:12]) or 'N/A'}",
        f"GO terms     : {', '.join(kb.get('go_terms', [])[:8]) or 'N/A'}",
        f"Cluster size : {kb.get('size', '?')} proteins",
        "",
        "--- KNOWN FUNCTIONS IN THIS CLUSTER ---",
    ]
    for fn in kb.get('sample_functions', [])[:2]:
        lines.append(f"  • {fn[:400]}")

    lines += ["", "--- TOP 3 MOST SIMILAR KNOWN PROTEINS ---"]
    for n in neighbors:
        d = n['data']
        name   = str(d.get('Protein names', 'Unknown'))[:80]
        func   = str(d.get('Function [CC]', 'N/A'))[:250]
        kw     = str(d.get('Keywords', 'N/A'))[:120]
        subfam = str(d.get('subfamily', 'N/A'))
        raw_cluster = int(d.get('super_cluster', -1))
        cluster_note = "(noise label)" if raw_cluster == -1 else f"cluster {raw_cluster}"
        lines += [
            f"\n  [{n['rank']}] {n['id']}  (cosine similarity: {n['similarity']:.4f})  [{cluster_note}]",
            f"      Name     : {name}",
            f"      Subfamily: {subfam}",
            f"      Keywords : {kw}",
            f"      Function : {func}",
        ]

    return "\n".join(lines), voted_cluster, confidence


def llm_answer(context: str, question: str, max_new_tokens: int = 350) -> str:
    system_msg = (
        "You are an expert computational biologist specialising in GPCR protein classification. "
        "Answer the user's question based strictly on the provided context. "
        "Be concise, scientifically accurate, and highlight key biological insights."
    )
    user_msg = f"CONTEXT:\n{context}\n\nQUESTION: {question}"

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_msg},
    ]
    prompt = llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(
        next(llm_model.parameters()).device
    )
    with torch.no_grad():
        out = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    generated = out[0][inputs['input_ids'].shape[1]:]
    return llm_tokenizer.decode(generated, skip_special_tokens=True).strip()


def analyze_with_llm(raw_sequence: str, question: str, k: int = 3) -> dict:

    import time
    t0 = time.time()

    cleaned = clean_sequence(raw_sequence)
    print(f"  Sequence: {len(raw_sequence)} chars → {len(cleaned)} AAs after cleaning")

    print(f"  Embedding cleaned sequence...")
    query_vec = embed_sequence(cleaned) 

    print(f"  Retrieving {k} nearest neighbors...")
    neighbors = retrieve_neighbors(query_vec, k=k)

    print(f"  Building cluster context...")
    context, cluster, confidence = build_rag_context(neighbors)

    print(f"  Generating LLM answer...")
    answer = llm_answer(context, question)

    elapsed = time.time() - t0
    print(f"  Done in {elapsed:.1f}s\n")

    return {
        'cluster': cluster,
        'confidence': confidence,
        'neighbors': neighbors,
        'context': context,
        'answer': answer,
    }



**Interactive Demo**

In [20]:
query_sequence = (
"QILWSIIFGTMVFVAAGGNIIVIWIVLTNKRMRTVTNYFLVNLSVADIMVSTLNVIFNFIYMLNSNWPFGELFCKITNFIAILSVGASVFTLMAISIDRYLAIVHPLRPRMSRTATVIIIVIIWIASSFLSLPNIICSKTIVEEFKNGDSRVVCYLEWYDGISTKSRIEYIYNVIILLVTYMLPIASMSYTYFRVGRELWGSQSIGECTAKQMESIKSKRKIVKMMMIVVAIFGVCWAPYHIYFLLAHHYPQIINSKYVQHTYLTIYWLAMSNSVYNPFVYCWMNSRFRQGFRNIFCC"
)

user_question = (
    "What function does this protein likely belong to, "
    "and what is its cluster?"
)

print("RAG-LLM PROTEIN ANALYSIS")
print("="*70)
print(f"Sequence length : {len(query_sequence)} amino acids")
print(f"Question        : {user_question}")
print("="*70 + "\n")

result = analyze_with_llm(query_sequence, user_question, k=3)

print("\n" + "─"*70)
print("RETRIEVED CONTEXT (sent to LLM):")
print("─"*70)
print(result['context'])

print("\n" + "─"*70)
print("LLM ANSWER:")
print("─"*70)
print(result['answer'])

label = "NOISE / unclassified" if result['cluster'] == -1 else f"Cluster {result['cluster']}"
print("\n" + "="*70)
print(f"Predicted cluster : {label}")
print(f"Confidence        : {result['confidence']:.0%}")
print(f"Top-1 similar ID  : {result['neighbors'][0]['id']} (sim={result['neighbors'][0]['similarity']:.4f})")
print("="*70)


RAG-LLM PROTEIN ANALYSIS
Sequence length : 298 amino acids
Question        : What function does this protein likely belong to, and what is its cluster?

  Sequence: 298 chars → 298 AAs after cleaning
  Embedding cleaned sequence...
Reloading ESM-2 for embedding ...


Loading weights: 100%|██████████| 566/566 [00:03<00:00, 172.87it/s, Materializing param=encoder.layer.32.output.dense.weight]                      
EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Retrieving 3 nearest neighbors...
  Building cluster context...
  Generating LLM answer...
  Done in 117.0s


──────────────────────────────────────────────────────────────────────
RETRIEVED CONTEXT (sent to LLM):
──────────────────────────────────────────────────────────────────────
=== PREDICTED GPCR CLUSTER: Cluster 16 ===
(Majority vote: 3/3 neighbors agree — 100% confidence)

--- CLUSTER BIOLOGICAL PROFILE ---
Subfamilies  : Other_ClassA
Keywords     : Cell membrane, G-protein coupled receptor, Membrane, Receptor, Reference proteome, Transducer, Transmembrane, Transmembrane helix
GO terms     : plasma membrane [GO:0005886], tachykinin receptor activity [GO:0004995], neuropeptide Y receptor activity [GO:0004983]
Cluster size : 6 proteins

--- KNOWN FUNCTIONS IN THIS CLUSTER ---

--- TOP 3 MOST SIMILAR KNOWN PROTEINS ---

  [1] A0A6P8XRN8  (cosine similarity: 0.7123)  [cluster 16]
      Name     : Tachykinin-like peptides receptor 99D isoform X2
      Subfamily: Other_ClassA
     